Step 2 – Upload processed datasets

In [1]:
from google.colab import files

uploaded = files.upload()

Saving portuguese_test_processed.csv to portuguese_test_processed.csv
Saving portuguese_train_processed.csv to portuguese_train_processed.csv


Step 3 – Import required libraries

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.dummy import DummyRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("All libraries imported successfully!")

All libraries imported successfully!


Step 4 – Create project folders

In [3]:
project_dir = Path("/content/Portuguese_ML_Project")

models_dir = project_dir / "models"

results_dir = (
    project_dir / "output" /
    "gradient_boosting" / "comparison"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully!")
print("Results directory:", results_dir)

Project folders created successfully!
Results directory: /content/Portuguese_ML_Project/output/gradient_boosting/comparison


Step 5 – Load processed Portuguese datasets

In [5]:
train_df = pd.read_csv(
    "/content/portuguese_train_processed.csv"
)

test_df = pd.read_csv(
    "/content/portuguese_test_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

X_test = test_df.drop(columns=["G3"])
y_test = test_df["G3"]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all()
assert y_train.notna().all()
assert X_test.notna().all().all()
assert y_test.notna().all()

display(train_df.head())

Training Shape: (519, 58)
Testing Shape: (130, 58)
Training Target: (519,)
Testing Target: (130,)


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,18
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,12
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,13
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,10


Step 6 – Define Gradient Boosting experiments

In [6]:
gb_experiments = {

    "GB-1": {
        "n_estimators": 100,
        "learning_rate": 0.10,
        "max_depth": 3,
        "min_samples_leaf": 1
    },

    "GB-2": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_samples_leaf": 2
    },

    "GB-3": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 2,
        "min_samples_leaf": 2
    },

    "GB-4": {
        "n_estimators": 150,
        "learning_rate": 0.10,
        "max_depth": 2,
        "min_samples_leaf": 4
    },

    "GB-5": {
        "n_estimators": 300,
        "learning_rate": 0.10,
        "max_depth": 4,
        "min_samples_leaf": 2
    }
}

experiment_df = pd.DataFrame.from_dict(
    gb_experiments,
    orient="index"
)

experiment_df.index.name = "Experiment"

print("GRADIENT BOOSTING EXPERIMENT CONFIGURATIONS")

display(experiment_df.reset_index())

GRADIENT BOOSTING EXPERIMENT CONFIGURATIONS


,Experiment,n_estimators,learning_rate,max_depth,min_samples_leaf
0,GB-1,100,0.10,3,1
1,GB-2,200,0.05,3,2
2,GB-3,300,0.03,2,2
3,GB-4,150,0.10,2,4
4,GB-5,300,0.10,4,2


Step 7 – Configure five-fold cross-validation

In [7]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("5-Fold Cross-Validation configured successfully!")

5-Fold Cross-Validation configured successfully!


Step 8 – Train and evaluate all Gradient Boosting configurations

In [8]:
gb_results = []
gb_fold_results = []
gb_predictions = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, params in gb_experiments.items():

    print(f"Training {name}...")

    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )

    # Five-fold cross-validation
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise"
    )

    # Out-of-fold predictions
    predictions = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv
    )

    gb_predictions[name] = predictions

    # Round predictions to whole grades
    rounded_predictions = np.clip(
        np.rint(predictions),
        0,
        20
    ).astype(int)

    # Convert numerical grades into pass/fail
    predicted_binary = (
        predictions >= 10
    ).astype(int)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    # Calculate training and validation RMSE
    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    # Save experiment results
    gb_results.append({

        "Experiment": name,

        "Estimators": params["n_estimators"],
        "Learning Rate": params["learning_rate"],
        "Max Depth": params["max_depth"],
        "Min Leaf": params["min_samples_leaf"],

        # Regression metrics
        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        # Grade prediction accuracy
        "Exact Grade Accuracy (%)": (
            np.mean(
                rounded_predictions == actual
            ) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        # Pass/fail classification metrics
        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        # Confusion matrix
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

    # Store fold-level results
    for fold in range(5):

        gb_fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train MSE": -scores["train_MSE"][fold],
            "Validation MSE": -scores["test_MSE"][fold],

            "Train RMSE": np.sqrt(
                -scores["train_MSE"][fold]
            ),

            "Validation RMSE": np.sqrt(
                -scores["test_MSE"][fold]
            ),

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

gb_results_df = pd.DataFrame(gb_results)

gb_results_df = gb_results_df.sort_values(
    by=[
        "Validation RMSE",
        "Validation MAE",
        "RMSE Gap"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

gb_fold_df = pd.DataFrame(gb_fold_results)

print("\nAll Gradient Boosting experiments completed!")

display(gb_results_df.round(4))

Training GB-1...
Training GB-2...
Training GB-3...
Training GB-4...
Training GB-5...

All Gradient Boosting experiments completed!


,Experiment,Estimators,Learning Rate,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,...,Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP
0,GB-3,300,0.03,2,2,0.6492,0.8577,0.8988,1.7709,0.9460,...,92.6782,90.9441,0.9685,0.9217,0.8471,0.9445,72,13,34,400
1,GB-4,150,0.10,2,4,0.5994,0.8825,0.7513,1.8407,0.8650,...,92.8709,90.5588,0.9730,0.9124,0.8706,0.9417,74,11,38,396
2,GB-2,200,0.05,3,2,0.4848,0.8710,0.4361,1.8991,0.6591,...,94.2197,90.1734,0.9776,0.9032,0.8941,0.9389,76,9,42,392
3,GB-1,100,0.10,3,1,0.4741,0.8948,0.3935,1.9685,0.6263,...,93.0636,89.9807,0.9823,0.8963,0.9176,0.9373,78,7,45,389
4,GB-5,300,0.10,4,2,0.0930,0.9427,0.0192,2.1656,0.1381,...,92.6782,88.8247,0.9796,0.8848,0.9059,0.9298,77,8,50,384


Step 9 – Regression error comparison

In [9]:
regression_columns = [
    "Experiment",
    "Estimators",
    "Learning Rate",
    "Max Depth",
    "Min Leaf",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 65)
print("GRADIENT BOOSTING – REGRESSION ERROR COMPARISON")
print("=" * 65)

display(
    gb_results_df[regression_columns].round(4)
)

GRADIENT BOOSTING – REGRESSION ERROR COMPARISON


,Experiment,Estimators,Learning Rate,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,GB-3,300,0.03,2,2,0.6492,0.8577,0.8988,1.7709,0.9460,1.3014,0.9148,0.8315,0.3554
1,GB-4,150,0.10,2,4,0.5994,0.8825,0.7513,1.8407,0.8650,1.3274,0.9287,0.8243,0.4625
2,GB-2,200,0.05,3,2,0.4848,0.8710,0.4361,1.8991,0.6591,1.3582,0.9586,0.8158,0.6991
3,GB-1,100,0.10,3,1,0.4741,0.8948,0.3935,1.9685,0.6263,1.3771,0.9626,0.8068,0.7508
4,GB-5,300,0.10,4,2,0.0930,0.9427,0.0192,2.1656,0.1381,1.4543,0.9982,0.7900,1.3162


Step 10 – Grade prediction accuracy comparison

In [10]:
grade_columns = [
    "Experiment",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 65)
print("GRADIENT BOOSTING – GRADE ACCURACY COMPARISON")
print("=" * 65)

display(
    gb_results_df[grade_columns].round(2)
)

GRADIENT BOOSTING – GRADE ACCURACY COMPARISON


,Experiment,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,GB-3,41.23,73.60,92.68
1,GB-4,39.11,71.48,92.87
2,GB-2,40.85,72.25,94.22
3,GB-1,40.85,70.71,93.06
4,GB-5,38.92,65.90,92.68


Step 11 – Pass/fail classification comparison

In [11]:
classification_columns = [
    "Experiment",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 65)
print("GRADIENT BOOSTING – CLASSIFICATION COMPARISON")
print("=" * 65)

display(
    gb_results_df[classification_columns].round(4)
)

GRADIENT BOOSTING – CLASSIFICATION COMPARISON


,Experiment,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,GB-3,90.9441,0.9685,0.9217,0.8471,0.9445
1,GB-4,90.5588,0.9730,0.9124,0.8706,0.9417
2,GB-2,90.1734,0.9776,0.9032,0.8941,0.9389
3,GB-1,89.9807,0.9823,0.8963,0.9176,0.9373
4,GB-5,88.8247,0.9796,0.8848,0.9059,0.9298


Step 12 – Confusion matrix comparison

In [12]:
confusion_columns = [
    "Experiment",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("=" * 65)
print("GRADIENT BOOSTING – CONFUSION MATRIX COMPARISON")
print("=" * 65)

display(
    gb_results_df[confusion_columns]
)

GRADIENT BOOSTING – CONFUSION MATRIX COMPARISON


,Experiment,TN,FP,FN,TP
0,GB-3,72,13,34,400
1,GB-4,74,11,38,396
2,GB-2,76,9,42,392
3,GB-1,78,7,45,389
4,GB-5,77,8,50,384


Step 13 – Check overfitting and underfitting

In [13]:
baseline_model = DummyRegressor(
    strategy="mean"
)

baseline_scores = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

gb_results_df["Fitting Assessment"] = (
    gb_results_df.apply(
        assess_fitting,
        axis=1
    )
)

print("=" * 65)
print("GRADIENT BOOSTING – FITTING ASSESSMENT")
print("=" * 65)

print(f"Baseline Validation RMSE: {baseline_rmse:.4f}")

display(
    gb_results_df[
        [
            "Experiment",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

GRADIENT BOOSTING – FITTING ASSESSMENT
Baseline Validation RMSE: 3.2326


,Experiment,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,GB-3,0.9460,1.3014,0.3554,Possible overfitting
1,GB-4,0.8650,1.3274,0.4625,Possible overfitting
2,GB-2,0.6591,1.3582,0.6991,Possible overfitting
3,GB-1,0.6263,1.3771,0.7508,Possible overfitting
4,GB-5,0.1381,1.4543,1.3162,Possible overfitting


Step 14 – Five-fold performance comparison

In [14]:
print("=" * 65)
print("GRADIENT BOOSTING – FIVE-FOLD PERFORMANCE")
print("=" * 65)

display(
    gb_fold_df.sort_values(
        ["Experiment", "Fold"]
    ).round(4)
)

GRADIENT BOOSTING – FIVE-FOLD PERFORMANCE


,Experiment,Fold,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2
0,GB-1,1,0.4773,0.7798,0.4149,1.5413,0.6441,1.2415,0.9625,0.8155
1,GB-1,2,0.4744,0.9666,0.3715,2.2935,0.6095,1.5144,0.9668,0.6986
2,GB-1,3,0.4723,0.8627,0.3970,1.7216,0.6301,1.3121,0.9602,0.8650
3,GB-1,4,0.4475,1.0922,0.3287,3.2638,0.5733,1.8066,0.9656,0.7642
4,GB-1,5,0.4990,0.7728,0.4552,1.0222,0.6747,1.0110,0.9579,0.8908
5,GB-2,1,0.4936,0.7916,0.4763,1.5715,0.6902,1.2536,0.9569,0.8119
6,GB-2,2,0.4860,0.8802,0.4199,1.8636,0.6480,1.3651,0.9625,0.7551
7,GB-2,3,0.4797,0.8283,0.4357,1.7849,0.6601,1.3360,0.9563,0.8600
8,GB-2,4,0.4538,1.0730,0.3460,3.1479,0.5882,1.7742,0.9638,0.7726
9,GB-2,5,0.5108,0.7821,0.5026,1.1274,0.7089,1.0618,0.9535,0.8796


Step 15 – Final complete comparison

In [15]:
final_comparison = gb_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL PORTUGUESE GRADIENT BOOSTING COMPARISON")
print("=" * 70)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(final_comparison.round(4))

FINAL PORTUGUESE GRADIENT BOOSTING COMPARISON


,Rank,Experiment,Estimators,Learning Rate,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP,Fitting Assessment
0,1,GB-3,300,0.03,2,2,0.6492,0.8577,0.8988,1.7709,0.9460,1.3014,0.9148,0.8315,0.3554,41.2331,73.6031,92.6782,90.9441,0.9685,0.9217,0.8471,0.9445,72,13,34,400,Possible overfitting
1,2,GB-4,150,0.10,2,4,0.5994,0.8825,0.7513,1.8407,0.8650,1.3274,0.9287,0.8243,0.4625,39.1137,71.4836,92.8709,90.5588,0.9730,0.9124,0.8706,0.9417,74,11,38,396,Possible overfitting
2,3,GB-2,200,0.05,3,2,0.4848,0.8710,0.4361,1.8991,0.6591,1.3582,0.9586,0.8158,0.6991,40.8478,72.2543,94.2197,90.1734,0.9776,0.9032,0.8941,0.9389,76,9,42,392,Possible overfitting
3,4,GB-1,100,0.10,3,1,0.4741,0.8948,0.3935,1.9685,0.6263,1.3771,0.9626,0.8068,0.7508,40.8478,70.7129,93.0636,89.9807,0.9823,0.8963,0.9176,0.9373,78,7,45,389,Possible overfitting
4,5,GB-5,300,0.10,4,2,0.0930,0.9427,0.0192,2.1656,0.1381,1.4543,0.9982,0.7900,1.3162,38.9210,65.8960,92.6782,88.8247,0.9796,0.8848,0.9059,0.9298,77,8,50,384,Possible overfitting


Step 16 – Select the best Gradient Boosting configuration

In [16]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_params = gb_experiments[best_name]

print("=" * 60)
print("BEST PORTUGUESE GRADIENT BOOSTING CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Parameters:", best_params)
print("Number of Features:", X_train.shape[1])

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy   : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision  : {best_result['Precision']:.4f}")
print(f"Recall     : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score   : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST PORTUGUESE GRADIENT BOOSTING CONFIGURATION
Experiment: GB-3
Parameters: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 2, 'min_samples_leaf': 2}
Number of Features: 58

REGRESSION METRICS
Validation RMSE : 1.3014
Validation MAE  : 0.8577
Validation R²   : 0.8315
RMSE Gap        : 0.3554

GRADE ACCURACY
Exact Accuracy  : 41.23%
Within ±1 Grade : 73.60%
Within ±2 Grades: 92.68%

PASS/FAIL CLASSIFICATION
Accuracy   : 90.94%
Precision  : 0.9685
Recall     : 0.9217
Specificity: 0.8471
F1 Score   : 0.9445

CONFUSION MATRIX
TN: 72
FP: 13
FN: 34
TP: 400

FITTING ASSESSMENT
Possible overfitting
